## 1. Data Audit

The objective of this phase is to understand the structure and quality
of the training and test datasets before performing EDA, feature
engineering, validation, or model training.

### Audit objectives

1. Dataset structure
2. Data types
3. Missing values
4. Cardinality
5. Duplicate observations
6. Train/test consistency
7. Target variable audit


In [ ]:
# Import and load data
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

train = pd.read_csv("../data/train.csv")
test = pd.read_csv("../data/test.csv")

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("Sum of Train duplicates", train.duplicated().sum())
train

In [ ]:
# Inspect Columns
print("Train Columns")

for i, col in enumerate(train.columns, start=1):
    print(f"{i:2}. {col}")


print("\nTest Columns")

for i, col in enumerate(test.columns, start=1):
    print(f"{i:2}. {col}")

In [ ]:
# Basic Structure
print("Train Info")
train.info()

print("\nTest Info")
test.info()

In [ ]:
# Data Type Audit
dtype_audit = pd.DataFrame({
    "dtype": train.dtypes.astype(str),
    "missing": train.isna().sum(),
    "missing_pct": (train.isna().mean() * 100).round(2),
    "unique_values": train.nunique(dropna=True)
})

dtype_audit

In [ ]:
#Missing-value Audit
missing_audit = pd.DataFrame(
    {
    "train_missing": train.isna().sum(),
    "train_missing_pct": (train.isna().mean() * 100).round(2),
    "test_missing": test.isna().sum(),
    "test_missing_pct": (test.isna().mean() * 100).round(2)
    }
)

missing_audit = missing_audit.sort_values(
    "train_missing",
    ascending=False
)

missing_audit

In [ ]:
#Total Missing Values
print("Total missing cells:")

print("Train:", train.isna().sum().sum())
print("Test :", test.isna().sum().sum())

In [ ]:
# Unique Value Audit
cardinality = pd.DataFrame(
    {
    "dtype": train.dtypes.astype(str),
    "n_unique": train.nunique(dropna=True),
    "unique_pct": (train.nunique(dropna=True) / len(train) * 100).round(2)
    }
    )

cardinality = cardinality.sort_values("n_unique", ascending = False)

cardinality

In [ ]:
# Exact duplicates
print("Exact Duplicate Rows:")

print("Train:", train.duplicated().sum())
print("Test :", test.duplicated().sum())

In [ ]:
# Product Store duplicates
pair_columns = ["product_code", "store_code"]

train_pair_duplicates = train.duplicated(subset=pair_columns).sum()

test_pair_duplicates = test.duplicated(
    subset=pair_columns
).sum()

print("Duplicate Product-Store Combinations")

print("Train:", train_pair_duplicates)
print("Test :", test_pair_duplicates)

In [ ]:
# Identify Categorical Columns
categorical_columns = train.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Categorical columns:")
print(categorical_columns)

In [ ]:
# Compare Categorical Domains and discover if categories exist in test but not train
for column in categorical_columns:

    train_values = set(train[column].dropna().unique())
    test_values = set(test[column].dropna().unique())

    test_only_values = test_values - train_values

    print(f"\n{column}")
    print("Train unique values:", len(train_values))
    print("Test unique values :", len(test_values))
    print("Test-only values   :", test_only_values)

In [ ]:
# Target Statistics
target = "total_sales"

print("TARGET:", target)
display(train[target].describe().to_frame().T)

In [ ]:
# Target Validity
print("Missing target values :", train[target].isna().sum())
print("Zero target values    :", (train[target] == 0).sum())
print("Negative target values:", (train[target] < 0).sum())
print("Unique target values  :", train[target].nunique())

In [ ]:
# Target Quantiles
target_quantiles = train[target].quantile(
    [ 0.00, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00]
)

target_quantiles

In [ ]:
# Dataset Contract
target = "total_sales"

# Target exists only in train
assert target in train.columns
assert target not in test.columns

# Train and test predictors must match
train_features = set(train.columns) - {target}
test_features = set(test.columns)

assert train_features == test_features

# IDs should be unique
assert train["id"].is_unique
assert test["id"].is_unique

# Target should not contain missing values
assert train[target].notna().all()

print("Target correctly identified.")
print("Train/test predictor columns match.")
print("Train IDs are unique.")
print("Test IDs are unique.")
print("Training target contains no missing values.")
print("\nDataset Contract Passed.")

In [ ]:
# Category integrity audit

category_columns = [
    "fat_content",
    "product_category",
    "store_code",
    "store_size",
    "store_location_tier",
    "store_format",
]

for col in category_columns:
    print(f"{col}")

    print("\nTrain:")
    display(train[col].value_counts(dropna=False).to_frame("count"))

    print("\nTest:")
    display(test[col].value_counts(dropna=False).to_frame("count"))

In [ ]:
# Check for possible whitespace/case inconsistencies

for col in [
    "fat_content",
    "product_category",
    "store_size",
    "store_location_tier",
    "store_format",
]:
    values = pd.concat(
        [train[col], test[col]],
        ignore_index=True
    ).dropna().astype(str)

    normalized = values.str.strip().str.lower()

    variants = pd.DataFrame({
        "raw": values,
        "normalized": normalized
    }).drop_duplicates()

    grouped = variants.groupby("normalized")["raw"].agg(list)

    inconsistent = grouped[grouped.str.len() > 1]

    print(f"{col}")

    if len(inconsistent) == 0:
        print("No obvious case/whitespace variants found.")
    else:
        display(inconsistent.to_frame("variants"))

In [ ]:
# Product code structure audit

print("Number of unique product codes:")
print(train["product_code"].nunique())

print("\nSample product codes:")
display(train["product_code"].head(20).to_frame())

print("\nProduct code length distribution:")
display(
    train["product_code"]
    .astype(str)
    .str.len()
    .value_counts()
    .sort_index()
    .to_frame("count")
)

print("\nProduct code prefix distribution:")
display(
    train["product_code"]
    .astype(str)
    .str[:2]
    .value_counts()
    .head(30)
    .to_frame("count")
)

In [ ]:
# Check whether product codes have a stable relationship with product categories

product_category_mapping = (
    train.groupby("product_code")["product_category"]
    .nunique()
)

print("Product codes associated with more than one raw category:")
print((product_category_mapping > 1).sum())

print("\nMaximum number of raw categories associated with one product code:")
print(product_category_mapping.max())

In [ ]:
# Product-level attribute consistency audit

product_attributes = [
    "product_weight_kg",
    "fat_content",
    "shelf_visibility",
    "product_category",
    "product_price",
]

for col in product_attributes:
    consistency = train.groupby("product_code")[col].nunique(dropna=False)

    print(col)
    print("Products with exactly one unique value:", (consistency == 1).sum())
    print("Products with more than one unique value:", (consistency > 1).sum())
    print("Maximum unique values for one product:", consistency.max())

In [ ]:
# Number of stores associated with each product

stores_per_product = train.groupby("product_code")["store_code"].nunique()

print("Distribution of number of stores per product:")
display(stores_per_product.value_counts().sort_index().to_frame("number_of_products"))

print("\nMaximum stores observed for one product:", stores_per_product.max())
print("Products appearing in all 10 stores:", (stores_per_product == 10).sum())

# Findings & Inferences

The data audit established the structural properties and quality of the
training and test datasets before any transformation, imputation,
encoding, or feature engineering was applied.

## Key Findings

### 1. Dataset structure

- The training set contains 6,818 observations and 13 columns.
- The test set contains 1,705 observations and 12 columns.
- The predictor columns are consistent between train and test.
- Train and test IDs are unique.
- There are no exact duplicate rows.
- There are no duplicate `product_code + store_code` combinations.
- The target variable, `total_sales`, is present only in the training set
  and contains no missing or invalid values.

### 2. Missing data

Missing values are concentrated in two predictors:

- `product_weight_kg`: approximately 18% missing
- `store_size`: approximately 28% missing

The missingness rates are similar between the training and test sets,
suggesting that missing values are part of the dataset structure rather
than an obvious train/test discrepancy.

The missing values will therefore be investigated further before deciding
how they should be handled.

### 3. Categorical data integrity

`product_category` contains systematic capitalization differences.
For example, the same semantic category appears as:

- `Fruits and Vegetables`
- `fruits and vegetables`
- `FRUITS AND VEGETABLES`

This creates artificial categorical levels. These representations
should be normalized during preprocessing so that semantically identical
categories are treated consistently.

Other major categorical variables, including `fat_content`, `store_code`,
`store_location_tier`, and `store_format`, do not show the same obvious
case/whitespace inconsistency.

### 4. Product-code structure

There are 1,555 unique `product_code` values.

The codes have a fixed length and share the common `PRD-` prefix, while
the remaining characters appear identifier-like. No obvious semantic
meaning can be inferred from the raw character positions during the audit.

Therefore, individual characters should not automatically be treated as
meaningful features. The predictive value of `product_code` itself will
be investigated empirically.

### 5. Product attributes are not uniformly fixed by product code

An important structural observation emerged when checking attributes
within each `product_code`.

Most product codes are associated with multiple observed values of:

- `product_weight_kg`
- `shelf_visibility`
- `product_price`

In contrast, `fat_content` is perfectly consistent within every
`product_code`.

`product_category` also varies for many product codes, but the earlier
category-integrity audit indicates that much of this variation is caused
by capitalization differences.

The variation in weight, visibility, and price suggests that these
variables may capture context beyond immutable product identity,
potentially including product-store or store-specific effects.

This is an inference rather than a confirmed causal interpretation.
The EDA phase will investigate whether these variables systematically
vary with store characteristics and whether product-store interactions
are useful for prediction.

### 6. Product-store coverage

Products do not appear in every store.

The number of stores associated with a product ranges from 1 to 9,
with no product appearing in all 10 stores.

Therefore, the dataset does not form a complete product × store matrix.
This structure will be considered when designing validation strategies
and investigating product-store relationships.

### 7. Target variable

`total_sales` is strictly positive and right-skewed:

- Mean: approximately 2,174.76
- Median: approximately 1,790.89
- Standard deviation: approximately 1,697.89
- Minimum: 32.70
- Maximum: 12,996.82

Because the competition metric is RMSE, large prediction errors in the
upper tail can have a substantial effect on model performance.

A log-transformed target may therefore be worth testing later, but it
will not be assumed to be superior. Any target transformation will be
evaluated using RMSE on the original sales scale.

## Modeling Implications

The audit suggests several questions for the next phases:

1. How strongly do store characteristics relate to sales?
2. How strongly do product characteristics relate to sales?
3. Are price, visibility, and weight systematically associated with
   particular stores or store formats?
4. Can missing `store_size` or `product_weight_kg` values be informed by
   other variables?
5. Does `product_code` provide useful predictive information despite its
   high cardinality?
6. Are product-store interactions important?
7. How should validation account for repeated products across stores?
8. Do nonlinear relationships justify tree-based boosting models?
9. Does transforming the target improve RMSE on the original sales scale?

These questions will guide the exploratory analysis, validation design,
feature engineering, and model experiments that follow.

> **Conclusion:** The dataset is structurally consistent between
> train and test, but contains meaningful missingness, high-cardinality
> product identifiers, categorical inconsistencies, and substantial
> variation in several product attributes. These characteristics warrant
> investigation before modeling rather than being handled with generic
> preprocessing assumptions.

# Exploratory Data Analysis

The data audit established the structure, quality, and consistency of the
training and test datasets without modifying the raw data.

This phase investigates the statistical distributions and relationships
within the data to identify patterns that can inform validation, feature
engineering, and model selection.

The analysis is hypothesis-driven. Each investigation should provide
evidence for or against a modeling assumption.

### EDA objectives

1. Understand the distribution of the target variable.
2. Investigate numerical feature distributions and relationships.
3. Examine categorical feature effects and group differences.
4. Investigate missing-value patterns.
5. Explore product-level and store-level structure.
6. Identify potential nonlinearities and interactions.
7. Generate statistically motivated feature-engineering hypotheses.

No feature engineering or model fitting will be performed until the
relevant exploratory evidence has been examined.

In [ ]:
# Target variable: descriptive statistics

target = train["total_sales"]

target_stats = target.describe(percentiles=[
    0.01, 0.05, 0.10,
    0.25, 0.50, 0.75,
    0.90, 0.95, 0.99
])

target_stats

In [ ]:
# Additional target statistics

target_summary = pd.Series({
    "mean": target.mean(),
    "median": target.median(),
    "std": target.std(),
    "variance": target.var(),
    "min": target.min(),
    "max": target.max(),
    "skewness": target.skew(),
    "kurtosis": target.kurt(),
    "iqr": target.quantile(0.75) - target.quantile(0.25),
    "coefficient_of_variation": target.std() / target.mean(),
    "unique_values": target.nunique()
})

target_summary

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.hist(
    target,
    bins="fd",
    edgecolor="black"
)

plt.title("Distribution of Total Sales")
plt.xlabel("Total Sales")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig("../Gallery/target_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Log-transformed target distribution

log_target = np.log1p(target)

plt.figure(figsize=(10, 6))

plt.hist(
    log_target,
    bins="fd",
    edgecolor="black"
)

plt.title("Distribution of Log-Transformed Total Sales")
plt.xlabel("log1p(Total Sales)")
plt.ylabel("Frequency")
plt.tight_layout()

plt.savefig("../Gallery/log_target_distribution.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
# Compare raw and log-transformed target distributions statistically

target_distribution_comparison = pd.DataFrame({
    "Raw total_sales": [
        target.mean(),
        target.std(),
        target.skew(),
        target.kurt(),
        target.quantile(0.01),
        target.quantile(0.50),
        target.quantile(0.99)
    ],
    "Log1p total_sales": [
        log_target.mean(),
        log_target.std(),
        log_target.skew(),
        log_target.kurt(),
        log_target.quantile(0.01),
        log_target.quantile(0.50),
        log_target.quantile(0.99)
    ]
}, index=[
    "Mean",
    "Std",
    "Skewness",
    "Kurtosis",
    "1st percentile",
    "Median",
    "99th percentile"
])

target_distribution_comparison

In [ ]:
# Target concentration across sales quantiles

target_bins = pd.qcut(
    target,
    q=[0, 0.50, 0.75, 0.90, 0.95, 0.99, 1.00],
    duplicates="drop"
)

target_tail_summary = (
    train.groupby(target_bins, observed=True)["total_sales"]
    .agg(
        count="size",
        mean="mean",
        median="median",
        min="min",
        max="max"
    )
)

target_tail_summary

In [ ]:
# Contribution of each target region to total observed sales

target_mass = (
    train.groupby(target_bins, observed=True)["total_sales"]
    .agg(
        count="size",
        total_sales="sum"
    )
)

target_mass["observation_share"] = (
    target_mass["count"] / len(train)
)

target_mass["sales_share"] = (
    target_mass["total_sales"] / target.sum()
)

target_mass

In [ ]:
# Prepare target-mass data for plotting

plot_data = (
    target_mass
    .rename_axis("target_group")
    .reset_index()
)

labels = [
    "Bottom 50%",
    "50-75%",
    "75-90%",
    "90-95%",
    "95-99%",
    "Top 1%"
]

plot_data["target_group_label"] = labels

plt.figure(figsize=(10, 6))

plt.bar(
    plot_data["target_group_label"],
    plot_data["sales_share"] * 100,
    edgecolor="black"
)

plt.title("Share of Total Sales Across Target Quantile Groups")
plt.xlabel("Target quantile group")
plt.ylabel("Share of total sales (%)")
plt.xticks(rotation=20)
plt.tight_layout()

plt.savefig(
    "../Gallery/target_sales_mass_by_quantile.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

## Target Variable Findings

The target variable `total_sales` exhibits substantial positive
skewness and dispersion.

- Mean sales are approximately 2,174.76, compared with a median of
  approximately 1,790.89.
- The raw target has skewness of approximately 1.15, confirming a
  pronounced right-skewed distribution.
- The standard deviation is approximately 1,697.89, giving a
  coefficient of variation of approximately 0.78.
- The upper tail extends to approximately 12,996.82, although only
  about 1% of observations exceed 7,261.
- The upper half of observations accounts for approximately 80% of
  the total observed sales value, while the lower half accounts for
  approximately 20%.
- The top 10% of observations account for approximately 26.8% of
  total observed sales.

Applying `log1p` substantially reduces the original positive
skewness, but produces moderate negative skewness rather than a
symmetric distribution. Therefore, the log transformation should not
be assumed to be superior based on distributional shape alone.

### Modeling implications

The target distribution motivates several hypotheses for later
experiments:

1. Raw-target and log-target models should both be evaluated.
2. Evaluation must remain on the original sales scale using RMSE.
3. Model residuals should later be examined across target quantiles.
4. Particular attention should be given to systematic
   underprediction or overprediction in the upper tail.
5. Target transformation should be treated as an empirical modeling
   decision rather than a preprocessing requirement.

In [ ]:
# Statistical profile of product_price

price = train["product_price"]

price_summary = pd.Series({
    "count": price.count(),
    "mean": price.mean(),
    "median": price.median(),
    "std": price.std(),
    "variance": price.var(),
    "min": price.min(),
    "1%": price.quantile(0.01),
    "5%": price.quantile(0.05),
    "25%": price.quantile(0.25),
    "50%": price.quantile(0.50),
    "75%": price.quantile(0.75),
    "95%": price.quantile(0.95),
    "99%": price.quantile(0.99),
    "max": price.max(),
    "skewness": price.skew(),
    "kurtosis": price.kurt(),
    "iqr": price.quantile(0.75) - price.quantile(0.25),
    "coefficient_of_variation": price.std() / price.mean(),
    "unique_values": price.nunique()
})

price_summary

In [ ]:
# Distribution of product_price

plt.figure(figsize=(10, 6))

plt.hist(
    train["product_price"],
    bins="fd",
    edgecolor="black"
)

plt.title("Distribution of Product Price")
plt.xlabel("Product price")
plt.ylabel("Frequency")

plt.tight_layout()

plt.savefig(
    "../Gallery/product_price_distribution.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()

In [ ]:
# Relationship between product price and total sales
# using price quantile bins

price_sales = train[["product_price", "total_sales"]].copy()

price_sales["price_quantile"] = pd.qcut(
    price_sales["product_price"],
    q=10,
    duplicates="drop"
)

price_sales_summary = (
    price_sales
    .groupby("price_quantile", observed=True)
    .agg(
        observations=("total_sales", "size"),
        mean_price=("product_price", "mean"),
        median_price=("product_price", "median"),
        mean_sales=("total_sales", "mean"),
        median_sales=("total_sales", "median"),
        std_sales=("total_sales", "std")
    )
)

price_sales_summary

In [ ]:
# Correlation between product price and total sales

price_sales_corr = pd.Series({
    "Pearson": train["product_price"].corr(
        train["total_sales"],
        method="pearson"
    ),
    "Spearman": train["product_price"].corr(
        train["total_sales"],
        method="spearman"
    )
})

price_sales_corr

In [ ]:
# Product price vs total sales

plt.figure(figsize=(10, 6))

plt.scatter(
    train["product_price"],
    train["total_sales"],
    edgecolor = "black",
    alpha=0.8,
    s=18
)

plt.title("Product Price vs Total Sales")
plt.xlabel("Product price")
plt.ylabel("Total sales")

plt.tight_layout()

plt.savefig(
    "../Gallery/product_price_vs_total_sales.png",
    dpi=200,
    bbox_inches="tight"
)

plt.show()